## Libraries Importation

In [1]:
import sqlite3
import re
import ollama
import pandas as pd
import json
import math

## Data Handling

In [2]:
data = pd.read_csv(r".\data\Red30 Tech US online retail sales.csv")
data.head()

,OrderNum,OrderDate,OrderType,CustomerType,CustName,CustState,ProdCategory,ProdNumber,ProdName,Quantity,Price,Discount,OrderTotal
0,1100934,9/1/2017,Wholesale,Business,Gusikowski Group,North Carolina,Blueprints,BP102,Bsquare Robot Blueprint,10,8.99,1.8,88.1
1,1100935,9/1/2017,Retail,Individual,Spencer Educators,Delaware,Drone Kits,DK204,BYOD-300,2,89.00,0.0,178.0
2,1100936,9/1/2017,Wholesale,Business,Schinner Inc.,Florida,Training Videos,TV801,Aerial Security,10,36.99,7.4,362.5
3,1100937,9/1/2017,Retail,Individual,Saxon Laviss,Virginia,Robot Kits,RK602,BYOR-1000,1,189.00,0.0,189.0
4,1100938,9/1/2017,Retail,Business,Wilderman Technologies,Texas,eBooks,EB502,Building Your First Robot,4,24.95,0.0,99.8


In [3]:
## Check data rows and cols
print(f" The data has {data.shape[0]} rows and {data.shape[1]} columns.")

## Get Columns names
col_names = data.columns.tolist()
print(f"Columns names are: {col_names}")

 The data has 4976 rows and 13 columns.
Columns names are: ['OrderNum', 'OrderDate', 'OrderType', 'CustomerType', 'CustName', 'CustState', 'ProdCategory', 'ProdNumber', 'ProdName', 'Quantity', 'Price', 'Discount', 'OrderTotal']


In [4]:
# Print ALL occurrences of duplicate rows
print(data[data.duplicated(keep=False)])

Empty DataFrame
Columns: [OrderNum, OrderDate, OrderType, CustomerType, CustName, CustState, ProdCategory, ProdNumber, ProdName, Quantity, Price, Discount, OrderTotal]
Index: []


In [5]:
# Returns the total count of missing values for each column
print(data.isna().sum())

OrderNum        0
OrderDate       0
OrderType       0
CustomerType    0
CustName        0
CustState       0
ProdCategory    0
ProdNumber      0
ProdName        0
Quantity        0
Price           0
Discount        0
OrderTotal      0
dtype: int64


## Build the Database

In [6]:
## Connect to SQLite
conn = sqlite3.connect('./data/Red30_Tech_US_online_retail_sales_DB.db')

## Store the DataFrame into a table named 'retail_sales'
data.to_sql('retail_sales', conn, if_exists='replace', index=False)

## Always close the connection when done
conn.close()

In [7]:
## Test the DB. Data reading
conn = sqlite3.connect('./data/Red30_Tech_US_online_retail_sales_DB.db')

# Write your standard SQL query
query = "SELECT * FROM retail_sales WHERE CustomerType = 'Business'"

# Run the query and load results directly back into a DataFrame
filtered_df = pd.read_sql_query(query, conn)
print(filtered_df)

conn.close()

      OrderNum  OrderDate  OrderType CustomerType                CustName  \
0      1100934   9/1/2017  Wholesale     Business        Gusikowski Group   
1      1100936   9/1/2017  Wholesale     Business           Schinner Inc.   
2      1100938   9/1/2017     Retail     Business  Wilderman Technologies   
3      1100939   9/2/2017  Wholesale     Business           Turcotte Corp   
4      1100943   9/3/2017  Wholesale     Business      Hettinger and Sons   
...        ...        ...        ...          ...                     ...   
2927   1105904  8/29/2019  Wholesale     Business            Gibson Group   
2928   1105905  8/29/2019  Wholesale     Business           Rau-Dickinson   
2929   1105906  8/29/2019  Wholesale     Business        Reichel and Sons   
2930   1105907  8/29/2019  Wholesale     Business               Wolff LLC   
2931   1105908  8/29/2019     Retail     Business            McClure Inc.   

           CustState     ProdCategory ProdNumber                   ProdName

## Querying the DB

In [8]:
## Instances Initialization
DB_FILE = './data/Red30_Tech_US_online_retail_sales_DB.db'
MODEL_NAME = 'gemma3:4b'   
EMBED_MODEL = 'nomic-embed-text-v2-moe'

db_schema = """
Table: retail_sales
Columns:
 - OrderNum
 - OrderDate
 - OrderType
 - CustomerType -- Explicit Values: 'Individual', 'Business'
 - CustName
 - CustState
 - ProdCategory
 - ProdNumber
 - ProdName
 - Quantity
 - Price
 - Discount
 - OrderTotal
"""

def init_cache_db():
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    
    # Tier 1 Cache Table (Exact text matching)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS query_cache (
            natural_question TEXT UNIQUE,
            generated_sql TEXT
        )
    """)
    
    # Tier 2 Cache Table (Vector/Semantic matching)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS semantic_query_cache (
            natural_question TEXT UNIQUE,
            embedding TEXT,
            generated_sql TEXT
        )
    """)
    conn.commit()
    conn.close()

def get_embedding(text):
    response = ollama.embeddings(model=EMBED_MODEL, prompt=text.lower().strip())
    return response['embedding']

def cosine_similarity(v1, v2):
    """Calculates true cosine similarity bounded strictly between -1.0 and 1.0."""
    dot_prod = sum(a * b for a, b in zip(v1, v2))
    mag1 = math.sqrt(sum(a * a for a in v1))
    mag2 = math.sqrt(sum(b * b for b in v2))
    if not mag1 or not mag2:
        return 0.0
    return dot_prod / (mag1 * mag2)

def get_exact_cached_sql(user_question):
    """Tier 1: High-speed local text match lookup (0 API calls)."""
    normalized_question = user_question.lower().strip()
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("SELECT generated_sql FROM query_cache WHERE natural_question = ?", (normalized_question,))
    row = cursor.fetchone()
    conn.close()
    
    if row:
        print("🚀 Tier 1 Cache Hit! Exact string matched instantly.")
        return row[0]
    return None

def get_semantic_cached_sql(user_question, threshold=0.88):
    """Tier 2: Mathematical vector embedding lookup fallback."""
    new_vector = get_embedding(user_question)
    
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("SELECT natural_question, embedding, generated_sql FROM semantic_query_cache")
    rows = cursor.fetchall()
    conn.close()
    
    for old_question, old_embed_json, sql_query in rows:
        old_vector = json.loads(old_embed_json)
        similarity = cosine_similarity(new_vector, old_vector)
        
        # Proper bounded math threshold check
        if similarity >= threshold:
            print(f"⚡ Tier 2 Semantic Cache Hit! Matches previous intent: '{old_question}' (Similarity: {similarity:.2f})")
            return sql_query
            
    return None

def save_to_dual_caches(user_question, sql_query):
    """Saves to both Tier 1 and Tier 2 persistent cache tables synchronously."""
    normalized_question = user_question.lower().strip()
    
    # 1. Generate vector data for Tier 2 Semantic Layer
    vector = get_embedding(user_question)
    vector_json = json.dumps(vector)
    
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    
    # Populate Tier 1 Exact Cache
    cursor.execute(
        "INSERT OR REPLACE INTO query_cache (natural_question, generated_sql) VALUES (?, ?)",
        (normalized_question, sql_query)
    )
    
    # Populate Tier 2 Semantic Cache
    cursor.execute(
        "INSERT OR REPLACE INTO semantic_query_cache (natural_question, embedding, generated_sql) VALUES (?, ?, ?)", 
        (normalized_question, vector_json, sql_query)
    )
    
    conn.commit()
    conn.close()
    print("💾 Query successfully written to Tier 1 and Tier 2 persistent caches.")

def clean_sql_output(raw_sql):
    """Cleans up markdown artifacts and quote characters common to gemma models."""
    clean_query = re.sub(r'```(?:sql)?', '', raw_sql)
    clean_query = clean_query.replace('"', '').replace('`', '')
    return clean_query.strip()

def ask_llm_for_sql(user_question, schema):
    system_prompt = (
        "You are a strict Text-to-SQL translator engine. "
        "Analyze the schema columns. Return ONLY executable raw SQL code. "
        "Never use double quotes (\") or backticks around columns or table names. "
        "Do not write markdown block fences, text labels, or comments."
    )
    user_prompt = f"Schema:\n{schema}\n\nQuestion: {user_question}\n\nSQL Query:"
    
    response = ollama.generate(
        model=MODEL_NAME,
        system=system_prompt,
        prompt=user_prompt,
        options={"temperature": 0.0} 
    )
    return response['response'].strip()

def llm_run_query(natural_language_question):
    # --- Execution Flow ---
    init_cache_db()

    # Target Natural Query
    #natural_language_question = "rank states from the most to the least sales with their total sales"

    # Flag to watch if we need to commit new cache records down the line
    needs_cache_update = False

    # Step 1: Query Tier 1 Exact Cache
    generated_sql = get_exact_cached_sql(natural_language_question)

    # Step 2: Query Tier 2 Semantic Cache if Tier 1 misses
    if not generated_sql:
        generated_sql = get_semantic_cached_sql(natural_language_question)
        if generated_sql:
            # If Tier 2 hits but Tier 1 missed, make sure to link this exact phrasing to Tier 1
            needs_cache_update = True

    # Step 3: Call local LLM engine if both tiers fail
    if not generated_sql:
        print(f"🤖 Cache Miss (Tier 1 & 2). Generating via local {MODEL_NAME} engine...")
        raw_llm_output = ask_llm_for_sql(natural_language_question, db_schema)
        generated_sql = clean_sql_output(raw_llm_output)
        needs_cache_update = True

    print(f"📜 SQL to execute:\n{generated_sql}\n")

    try:
        # Step 4: Extract Dataframe Matrix via Database Connect
        conn = sqlite3.connect(DB_FILE)
        df_result = pd.read_sql_query(generated_sql, conn)
        conn.close()
        
        print("📊 Query Results:")
        print(df_result)
        
        # Step 5: ONLY write to persistent cache if execution completely succeeded
        if needs_cache_update:
            save_to_dual_caches(natural_language_question, generated_sql)

    except Exception as e:
        print(f"❌ Execution Failure: The query failed to execute. Cache was not updated. Error details: {e}")


In [9]:
llm_run_query("rank states from the most to the least sales with their total sales")

🤖 Cache Miss (Tier 1 & 2). Generating via local gemma3:4b engine...
📜 SQL to execute:
SELECT
  CustState,
  SUM(OrderTotal) AS TotalSales
FROM retail_sales
GROUP BY
  CustState
ORDER BY
  TotalSales DESC

📊 Query Results:
               CustState  TotalSales
0               New York   616925.84
1             California   540285.52
2                Florida   394483.74
3                  Texas   349925.48
4         North Carolina   345118.26
5           Pennsylvania   290120.09
6              Minnesota   267308.95
7             Washington   257140.80
8               Virginia   248797.89
9                Georgia   237607.26
10             Tennessee   206698.72
11                  Ohio   195307.10
12               Indiana   189361.69
13                 Idaho   175549.42
14              Colorado   174726.32
15              Michigan   154327.39
16         Massachusetts   152747.83
17              Maryland   144339.77
18               Alabama   137984.65
19              Oklahoma   132976.55
2

In [19]:
llm_run_query("top 5 of states with the most sales with their total sales")

🤖 Cache Miss (Tier 1 & 2). Generating via local gemma3:4b engine...
📜 SQL to execute:
SELECT
  CustState,
  SUM(OrderTotal) AS TotalSales
FROM retail_sales
GROUP BY
  CustState
ORDER BY
  TotalSales DESC
LIMIT 5

📊 Query Results:
        CustState  TotalSales
0        New York   616925.84
1      California   540285.52
2         Florida   394483.74
3           Texas   349925.48
4  North Carolina   345118.26
💾 Query successfully written to Tier 1 and Tier 2 persistent caches.


In [10]:
llm_run_query("total sales for retails order")

🤖 Cache Miss (Tier 1 & 2). Generating via local gemma3:4b engine...
📜 SQL to execute:
SELECT sum(OrderTotal) FROM retail_sales

📊 Query Results:
   sum(OrderTotal)
0       6899234.05
💾 Query successfully written to Tier 1 and Tier 2 persistent caches.


In [13]:
llm_run_query(" 'Drone Kits' total sales")

🤖 Cache Miss (Tier 1 & 2). Generating via local gemma3:4b engine...
📜 SQL to execute:
SELECT sum(OrderTotal) FROM retail_sales WHERE ProdCategory = 'Drone Kits'

📊 Query Results:
   sum(OrderTotal)
0        682597.11
💾 Query successfully written to Tier 1 and Tier 2 persistent caches.


In [17]:
llm_run_query(" total transaction count for individual")

🤖 Cache Miss (Tier 1 & 2). Generating via local gemma3:4b engine...
📜 SQL to execute:
SELECT COUNT(*) FROM retail_sales WHERE CustomerType = 'Individual'

📊 Query Results:
   COUNT(*)
0      2044
💾 Query successfully written to Tier 1 and Tier 2 persistent caches.


In [18]:
llm_run_query(" total transaction count for business")

🤖 Cache Miss (Tier 1 & 2). Generating via local gemma3:4b engine...
📜 SQL to execute:
SELECT COUNT(*) FROM retail_sales WHERE CustomerType = 'Business'

📊 Query Results:
   COUNT(*)
0      2932
💾 Query successfully written to Tier 1 and Tier 2 persistent caches.
